# Example Data Fingerprinting

In [1]:
from ncorr.ncorr import NCorrFP
import pandas as pd

## Fingerprint embedding process:
### Define parameters
- gamma is the embedding ratio. This means that probability of marking one record is 1/gamma
- fp_len is the fingerprint length in bits. The longer, the more secure, however if it is too long relative to data, it might not be robust. Rule of a thumb is #data_records/(gamma*fp_len)>20.
- user_id is an integer identifier associated with a particular data recipient
- infile: csv to be fingerprinted
- outfile: path to save the fingerprinted data

In [2]:
gamma = 11  # the embedding ratio: probability of marking one record is 1/gamma
fp_len = 256
user_id = 10
infile = "speml/Financial_Records_Demo.csv"
outfile = "speml/fingerprinted_output.csv"

### Defining the scheme

In [3]:
scheme = NCorrFP(gamma=gamma, fingerprint_bit_length=fp_len)

### Use secret key
For simulation purposes, the secret key is defined as an argument for fingerprint embedding. In real applications, the secret key is securely stored and accessed only by the owner.

In [4]:
secret_key = 177264  

### Embedding the fingerprint

In [5]:
fingerprinted_data = scheme.insertion(infile, 
                                      secret_key=secret_key, 
                                      recipient_id=user_id,
                                      outfile=outfile)

Start the NCorr fingerprint insertion algorithm. This might take a few minutes ...
	gamma: 11
	k: 50
	correlated attributes: []
Inserting the fingerprint...

Fingerprint inserted.
Runtime: 14.34 sec.


## Fingerprint detection process:
Fingerprint detection process requires defining the scheme with exact parameters used in the embedding (we use _scheme_ object for that). Furthermore, fingerprint can only be successfully extracted with the correct key.

Detecting the fingerprint from fingerprinted data:

In [6]:
suspect = scheme.detection(fingerprinted_data, 
                           secret_key=secret_key)
no1sus, no1sus_conf = max(suspect[2].items(), key=lambda item: item[1])
print("User id {} with confidence {}. ".format(no1sus, no1sus_conf))

Start NCorr fingerprint detection algorithm. This might take a few minutes ...
	gamma: 11
	k: 50
	fp length: 256
	max # recipients: 100
	correlated attributes: [Index(['person_id', 'age', 'gender', 'region', 'annual_income', 'credit_score',
       'loan_amount', 'monthly_expense', 'savings_balance',
       'investment_amount', 'employment_status', 'marital_status',
       'home_ownership', 'credit_card_usage', 'num_credit_cards',
       'loan_purpose', 'employment_years', 'debt_to_income_ratio',
       'late_payment_history', 'savings_rate', 'housing_market_exposure',
       'stock_market_exposure', 'bankruptcy_history',
       'monthly_loan_installment', 'loan_default'],
      dtype='object')]
Fingerprint detected: 0101100111000110000000000100101100111010001010001000011001010010111100110010010110000111111110100011001100100111111011011010001001101100111010010111110000100111111001110011011111011011111111101111001100001000111011001110001100101110010111001010111101010101
Runtime: 15.13 se

## Some examples that should fail:
### 1. Detecting the fingerprint from a non-fingerprinted data (false attribution):

In [7]:
non_fingerprinted_data = pd.read_csv(infile)
suspect = scheme.detection(non_fingerprinted_data, 
                           secret_key=secret_key)
no1sus, no1sus_conf = max(suspect[2].items(), key=lambda item: item[1])
print("User id {} with confidence {}. ".format(no1sus, no1sus_conf))

Start NCorr fingerprint detection algorithm. This might take a few minutes ...
	gamma: 11
	k: 50
	fp length: 256
	max # recipients: 100
	correlated attributes: [Index(['person_id', 'age', 'gender', 'region', 'annual_income', 'credit_score',
       'loan_amount', 'monthly_expense', 'savings_balance',
       'investment_amount', 'employment_status', 'marital_status',
       'home_ownership', 'credit_card_usage', 'num_credit_cards',
       'loan_purpose', 'employment_years', 'debt_to_income_ratio',
       'late_payment_history', 'savings_rate', 'housing_market_exposure',
       'stock_market_exposure', 'bankruptcy_history',
       'monthly_loan_installment', 'loan_default'],
      dtype='object')]
Fingerprint detected: 0100110102000000112011120112101021021101112000001100001000000101010211110100001211111001111001010200011101000100111002010012211001101110110021000012110110201100001111001100010111110010210021020010110221100111110000100120011020101001110011111010111100100100
Runtime: 15.75 se

### 2. Detection with a false secret key

In [8]:
false_key = 123
suspect = scheme.detection(fingerprinted_data, 
                           secret_key=false_key)
no1sus, no1sus_conf = max(suspect[2].items(), key=lambda item: item[1])
print("User id {} with confidence {}. ".format(no1sus, no1sus_conf))

Start NCorr fingerprint detection algorithm. This might take a few minutes ...
	gamma: 11
	k: 50
	fp length: 256
	max # recipients: 100
	correlated attributes: [Index(['person_id', 'age', 'gender', 'region', 'annual_income', 'credit_score',
       'loan_amount', 'monthly_expense', 'savings_balance',
       'investment_amount', 'employment_status', 'marital_status',
       'home_ownership', 'credit_card_usage', 'num_credit_cards',
       'loan_purpose', 'employment_years', 'debt_to_income_ratio',
       'late_payment_history', 'savings_rate', 'housing_market_exposure',
       'stock_market_exposure', 'bankruptcy_history',
       'monthly_loan_installment', 'loan_default'],
      dtype='object')]
Fingerprint detected: 0022002200111101011100120201101001000012002112000121120011101012211100210000111000012210100011000210011000120011201220011121011102201212112110002201201120010120110000100101200100110101100100001110010120000100011111100100121101201001000201012021012110001002
Runtime: 14.62 se

In [9]:
another_false_key = 177265
suspect = scheme.detection(fingerprinted_data, 
                           secret_key=another_false_key)
no1sus, no1sus_conf = max(suspect[2].items(), key=lambda item: item[1])
print("User id {} with confidence {}. ".format(no1sus, no1sus_conf))

Start NCorr fingerprint detection algorithm. This might take a few minutes ...
	gamma: 11
	k: 50
	fp length: 256
	max # recipients: 100
	correlated attributes: [Index(['person_id', 'age', 'gender', 'region', 'annual_income', 'credit_score',
       'loan_amount', 'monthly_expense', 'savings_balance',
       'investment_amount', 'employment_status', 'marital_status',
       'home_ownership', 'credit_card_usage', 'num_credit_cards',
       'loan_purpose', 'employment_years', 'debt_to_income_ratio',
       'late_payment_history', 'savings_rate', 'housing_market_exposure',
       'stock_market_exposure', 'bankruptcy_history',
       'monthly_loan_installment', 'loan_default'],
      dtype='object')]
Fingerprint detected: 1111112010010200010020010202001111012000000001101001020021021210000102000110001000101001211010110112001110200112002200112220121010100100010211110101001000220010211121101121001211000210110000122002010011000000112110210000100001001000000011111010101011212121
Runtime: 15.04 se

### 3. Detection using different parameter settings (but correct secret key)

In [10]:
slightly_different_scheme = NCorrFP(gamma=gamma-1, fingerprint_bit_length=fp_len)
suspect = slightly_different_scheme.detection(fingerprinted_data, 
                                              secret_key=secret_key)
no1sus, no1sus_conf = max(suspect[2].items(), key=lambda item: item[1])
print("User id {} with confidence {}. ".format(no1sus, no1sus_conf))

Start NCorr fingerprint detection algorithm. This might take a few minutes ...
	gamma: 10
	k: 50
	fp length: 256
	max # recipients: 100
	correlated attributes: [Index(['person_id', 'age', 'gender', 'region', 'annual_income', 'credit_score',
       'loan_amount', 'monthly_expense', 'savings_balance',
       'investment_amount', 'employment_status', 'marital_status',
       'home_ownership', 'credit_card_usage', 'num_credit_cards',
       'loan_purpose', 'employment_years', 'debt_to_income_ratio',
       'late_payment_history', 'savings_rate', 'housing_market_exposure',
       'stock_market_exposure', 'bankruptcy_history',
       'monthly_loan_installment', 'loan_default'],
      dtype='object')]
Fingerprint detected: 0101100111000110000000000100101100111010001010001000011001010010111100110010010110000111111110100011001100100111111011011010001001101100211010010111110000100111111001110011011111011011111111101111001100001000111011001112001100101110010111001010111101010101
Runtime: 16.42 se

In [11]:
very_different_scheme = NCorrFP(gamma=5, fingerprint_bit_length=200)
suspect = very_different_scheme.detection(fingerprinted_data, 
                                              secret_key=secret_key)
no1sus, no1sus_conf = max(suspect[2].items(), key=lambda item: item[1])
print("User id {} with confidence {}. ".format(no1sus, no1sus_conf))

Start NCorr fingerprint detection algorithm. This might take a few minutes ...
	gamma: 5
	k: 50
	fp length: 200
	max # recipients: 100
	correlated attributes: [Index(['person_id', 'age', 'gender', 'region', 'annual_income', 'credit_score',
       'loan_amount', 'monthly_expense', 'savings_balance',
       'investment_amount', 'employment_status', 'marital_status',
       'home_ownership', 'credit_card_usage', 'num_credit_cards',
       'loan_purpose', 'employment_years', 'debt_to_income_ratio',
       'late_payment_history', 'savings_rate', 'housing_market_exposure',
       'stock_market_exposure', 'bankruptcy_history',
       'monthly_loan_installment', 'loan_default'],
      dtype='object')]
Fingerprint detected: 11000011111121020112012111101111121011001011010011111000111201110001001010100112200020110101111101101010210100111112010011210120101101112111101010001201101000121011010001100101011012101100100010111211
Runtime: 26.09 sec.
User id 53 with confidence 0.55. 
